# Project #3 MEMM NER (Kaggle Ready)

这个 notebook 可以直接上传到 Kaggle 的课程比赛环境中运行。

运行完成后会在 `/kaggle/working/` 生成：
- `model.pkl`
- `submission.csv`


In [ ]:
!pip install -q nltk scikit-learn


In [ ]:
import pickle
import string
from pathlib import Path

import pandas as pd
from nltk.classify.maxent import MaxentClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score

LABELS = ["B-LOC", "I-LOC", "B-ORG", "I-ORG", "B-PER", "I-PER", "B-MISC", "I-MISC", "O"]
DATASET = Path("/kaggle/input/mem-classification")
WORKDIR = Path("/kaggle/working")

train_df = pd.read_csv(DATASET / "train.csv")
dev_df = pd.read_csv(DATASET / "dev.csv")
test_df = pd.read_csv(DATASET / "test.csv")

print("train rows:", len(train_df))
print("dev rows:", len(dev_df))
print("test rows:", len(test_df))


In [ ]:
def word_shape(token):
    chars = []
    for ch in token:
        if ch.isupper():
            chars.append("X")
        elif ch.islower():
            chars.append("x")
        elif ch.isdigit():
            chars.append("d")
        else:
            chars.append(ch)

    compact = []
    for ch in chars:
        if not compact or compact[-1] != ch:
            compact.append(ch)
    return "".join(compact)


def df_to_sentences(df, has_label=True):
    sentences = []
    for _, group in df.groupby("sentence_id", sort=True):
        ordered = group.sort_values("token_idx")
        tokens = ordered["token"].fillna("").astype(str).tolist()
        if has_label:
            labels = ordered["label"].astype(str).tolist()
            sentences.append((tokens, labels))
        else:
            sentences.append(tokens)
    return sentences


def flatten_labels(sentences):
    return [label for _, labels in sentences for label in labels]


def flatten_predictions(predictions):
    return [label for sentence in predictions for label in sentence]


In [ ]:
class MEMM:
    def __init__(self, max_iter=20):
        self.max_iter = max_iter
        self.classifier = None

    def features(self, words, prev_label, pos):
        word = "" if pd.isna(words[pos]) else str(words[pos])
        word_lower = word.lower()
        prev_word = "" if pos == 0 else str(words[pos - 1])
        next_word = "" if pos == len(words) - 1 else str(words[pos + 1])
        prev_lower = prev_word.lower()
        next_lower = next_word.lower()

        feats = {
            "bias": 1,
            f"word={word}": 1,
            f"word.lower={word_lower}": 1,
            f"shape={word_shape(word)}": 1,
            f"prev_label={prev_label}": 1,
            f"prev_word.lower={prev_lower}": 1,
            f"next_word.lower={next_lower}": 1,
            f"prev_label+word.lower={prev_label}|{word_lower}": 1,
            f"prev_word+word={prev_lower}|{word_lower}": 1,
            f"word+next_word={word_lower}|{next_lower}": 1,
            f"len={min(len(word), 10)}": 1,
        }

        if pos == 0:
            feats["BOS"] = 1
        if pos == len(words) - 1:
            feats["EOS"] = 1

        if word:
            feats[f"prefix1={word_lower[:1]}"] = 1
            feats[f"prefix2={word_lower[:2]}"] = 1
            feats[f"prefix3={word_lower[:3]}"] = 1
            feats[f"prefix4={word_lower[:4]}"] = 1
            feats[f"suffix1={word_lower[-1:]}"] = 1
            feats[f"suffix2={word_lower[-2:]}"] = 1
            feats[f"suffix3={word_lower[-3:]}"] = 1
            feats[f"suffix4={word_lower[-4:]}"] = 1

            if word[0].isupper():
                feats["is_title_init"] = 1
            if word.istitle():
                feats["is_title"] = 1
            if word.isupper():
                feats["is_upper"] = 1
            if word.islower():
                feats["is_lower"] = 1
            if word.isdigit():
                feats["is_digit"] = 1
            if any(ch.isdigit() for ch in word):
                feats["has_digit"] = 1
            if "-" in word:
                feats["has_hyphen"] = 1
            if "'" in word:
                feats["has_apostrophe"] = 1
            if "." in word:
                feats["has_period"] = 1
            if all(ch in string.punctuation for ch in word):
                feats["is_punct"] = 1

        if prev_word:
            feats[f"prev_shape={word_shape(prev_word)}"] = 1
            if prev_word.istitle():
                feats["prev_is_title"] = 1

        if next_word:
            feats[f"next_shape={word_shape(next_word)}"] = 1
            if next_word.istitle():
                feats["next_is_title"] = 1

        return feats

    def train(self, train_sents):
        samples = []
        for tokens, labels in train_sents:
            prev = "O"
            for i, label in enumerate(labels):
                samples.append((self.features(tokens, prev, i), label))
                prev = label

        self.classifier = MaxentClassifier.train(samples, max_iter=self.max_iter)

    def predict_sentence(self, tokens):
        preds = []
        prev = "O"
        for i in range(len(tokens)):
            label = self.classifier.classify(self.features(tokens, prev, i))
            preds.append(label)
            prev = label
        return preds

    def predict_corpus(self, sentences):
        return [self.predict_sentence(tokens) for tokens in sentences]


In [ ]:
train_sents = df_to_sentences(train_df, has_label=True)
dev_sents = df_to_sentences(dev_df, has_label=True)
test_sents = df_to_sentences(test_df, has_label=False)

model = MEMM(max_iter=20)
model.train(train_sents)

dev_gold = flatten_labels(dev_sents)
dev_preds = flatten_predictions(model.predict_corpus([tokens for tokens, _ in dev_sents]))

accuracy = accuracy_score(dev_gold, dev_preds)
precision = precision_score(dev_gold, dev_preds, average="macro", labels=LABELS, zero_division=0)
recall = recall_score(dev_gold, dev_preds, average="macro", labels=LABELS, zero_division=0)
f1 = f1_score(dev_gold, dev_preds, average="macro", labels=LABELS, zero_division=0)

print("Dev Accuracy :", round(accuracy, 4))
print("Dev Precision:", round(precision, 4))
print("Dev Recall   :", round(recall, 4))
print("Dev Macro F1 :", round(f1, 4))
print()
print(classification_report(dev_gold, dev_preds, labels=LABELS, zero_division=0))


In [ ]:
full_df = pd.concat([train_df, dev_df], ignore_index=True)
full_sents = df_to_sentences(full_df, has_label=True)

final_model = MEMM(max_iter=20)
final_model.train(full_sents)

test_preds = flatten_predictions(final_model.predict_corpus(test_sents))
assert len(test_preds) == len(test_df), "Prediction length mismatch with test.csv."

submission = pd.DataFrame({"id": test_df["id"], "label": test_preds})
submission.to_csv(WORKDIR / "submission.csv", index=False, encoding="utf-8")

with open(WORKDIR / "model.pkl", "wb") as f:
    pickle.dump(final_model, f)

print("Saved:", WORKDIR / "submission.csv")
print("Saved:", WORKDIR / "model.pkl")
submission.head()


提交说明：运行完成后，Kaggle 会在 `Output` 中看到 `submission.csv`。点击 `Save Version` 后即可提交到比赛。
